# Exercise 1: IEEE 802.15.4

Pietro Pizzoccheri 10797420 [team leader]  
Lorenzo Bardelli 10831941

## Data
- Poisson peaks per 10 s window: $\lambda = 0.4$
- Window duration: $T_w = 10$ s
- PHY bitrate: $R = 250$ kbps
- Packet size: $L = 128$ bytes
- Number of vibration nodes: 3
- Data sent per window:
  - 0 peaks -> 512 bytes
  - 1 peak -> 2 KB = 2048 bytes
  - more than 1 peak -> 4 KB = 4096 bytes

$1$ KB = $1024$ bytes.

## Q1 : Compute the Probability Mass Function of the output rate: 𝑃(𝑟=𝑟0),𝑃(𝑟=𝑟1),𝑃(𝑟=𝑟2)
where 𝑟0, 𝑟1, and 𝑟2 are the output rates when the node detects 0, 1, or more than 1
vibration peaks, respectively.

In [24]:
import math

lam = 0.4
Tw = 10.0
R = 250_000  # bit/s
L_bytes = 128
nodes = 3
payload_0_bytes = 512
payload_1_bytes = 2 * 1024
payload_2_bytes = 4 * 1024

# Poisson probabilities for 0, 1, and more than 1 detected peaks.
p0 = math.exp(-lam)
p1 = lam * math.exp(-lam)
pgt1 = 1 - p0 - p1

# Output rates
r0 = payload_0_bytes / Tw
r1 = payload_1_bytes / Tw
r2 = payload_2_bytes / Tw

print('Output rates:')
print(f'r0 = {r0:.1f} B/s')
print(f'r1 = {r1:.1f} B/s')
print(f'r2 = {r2:.1f} B/s')
print()
print('PMF of output rate:')
print(f'P(r = r0)  = {p0:.4f}')
print(f'P(r = r1) = {p1:.4f}')
print(f'P(r = r2) = {pgt1:.4f}')


Output rates:
r0 = 51.2 B/s
r1 = 204.8 B/s
r2 = 409.6 B/s

PMF of output rate:
P(r = r0)  = 0.6703
P(r = r1) = 0.2681
P(r = r2) = 0.0616


## Q2 : Based on the output rate PMF, compute a consistent CFP slot assignment for the system.

Following the reference exercise, dimension the beacon interval so that one CFP slot per beacon interval supports the minimum output rate: $BI=L/r_{min}$. The worst-case slots per node are then $\lceil r_{max}/(L/BI) \rceil$.

In [25]:
# Packets needed in each case.
packets_per_window_0 = math.ceil(payload_0_bytes / L_bytes)
packets_per_window_1 = math.ceil(payload_1_bytes / L_bytes)
packets_per_window_2 = math.ceil(payload_2_bytes / L_bytes)

# Dimensioning BI so that one CFP slot per BI carries the minimum output rate.
L_bits = L_bytes * 8
rates = [r0, r1, r2]
r_min = min(rates)
r_max = max(rates)
BI = L_bytes / r_min
slot_rate_bytes = L_bytes / BI

# Worst-case CFP sizing
slots_per_node = math.ceil(r_max / slot_rate_bytes)
beacon_intervals_per_window = Tw / BI
total_cfp_slots = slots_per_node * nodes

print(f'Minimum rate r_min = {r_min:.1f} B/s')
print(f'Slot payload rate = {slot_rate_bytes:.1f} B/s')
print(f'Worst-case packets per node = {packets_per_window_2}')
print(f'Beacon interval BI = {BI:.3f} s')
print(f'Beacon intervals per 10 s window = {beacon_intervals_per_window:.0f}')
print(f'CFP slots per node per BI = {slots_per_node}')
print(f'Total CFP slots per BI ({nodes} nodes) = {total_cfp_slots}')


Minimum rate r_min = 51.2 B/s
Slot payload rate = 51.2 B/s
Worst-case packets per node = 32
Beacon interval BI = 2.500 s
Beacon intervals per 10 s window = 4
CFP slots per node per BI = 8
Total CFP slots per BI (3 nodes) = 24


## Q3: Compute:
- $T_s$
- $N_{slots}$ in the CFP
- $T_{active}$
- $T_{inactive}$
- The duty cycle of the system

Use $T_s=8L/R$ because $L$ is in bytes and $R$ is in bit/s, $T_{CFP}=N_{CFP}T_s$, $T_{active}=T_{CFP}+T_s$ for the beacon slot, $T_{inactive}=BI-T_{active}$, and $\eta=T_{active}/BI$.

In [26]:
# slot time, active time, inactive time, duty cycle
Ts = (L_bytes * 8) / R  # seconds
Tcfp = total_cfp_slots * Ts
Tbeacon = Ts
Tactive = Tcfp + Tbeacon
Tinactive = BI - Tactive
DC = Tactive / BI

print(f'BI        = {BI:.3f} s')
print(f'Ts        = {Ts*1000:.3f} ms')
print(f'N_CFP     = {total_cfp_slots} slots')
print(f'T_CFP     = {Tcfp*1000:.3f} ms')
print(f'Tactive   = {Tactive*1000:.3f} ms')
print(f'Tinactive = {Tinactive:.6f} s')
print(f'DutyCycle = {DC*100:.3f}%')

BI        = 2.500 s
Ts        = 4.096 ms
N_CFP     = 24 slots
T_CFP     = 98.304 ms
Tactive   = 102.400 ms
Tinactive = 2.397600 s
DutyCycle = 4.096%


## Q4: How many additional vibration monitoring nodes can be added while keeping the duty cycle below 10%?

With the same slot assignment, solve $(N_{nodes}\cdot slots_{node}+1)T_s/BI < 10\%$.

In [27]:
# how many nodes can we support with duty cycle < 10%?
duty_cycle_limit = 0.10
max_active_slots = (duty_cycle_limit * BI) / Ts
max_cfp_slots = math.ceil(max_active_slots - 1) - 1  # strict < limit, reserve 1 beacon slot
N_nodes_max = max_cfp_slots // slots_per_node
N_additional = N_nodes_max - nodes

print(f'Max nodes with DC<10% = {N_nodes_max}')
print(f'Additional nodes      = {N_additional}')

Max nodes with DC<10% = 7
Additional nodes      = 4


## Final answers
- $P(r_0)=0.6703$, $P(r_1)=0.2681$, $P(r_2)=0.0616$
- $BI = 2.5$ s.
- CFP slots per node per beacon interval: **8**
- Total CFP slots per beacon interval for 3 nodes: **24**
- Slot time: **4.096 ms**
- CFP duration: **98.304 ms**
- Active time (with beacon slot): **102.400 ms**
- Inactive time: **2397.600 ms**
- Duty cycle: **4.096%**
- Additional nodes while keeping duty cycle below 10%: **4**
